Continuing SPTSSB rs1450522 associations with PD phenotypes
## 2. Read processed data in R

### First, clinical data

In [ ]:
#install.packages("nloptr", verbose = TRUE)
#Best way to install (because R is installed in a conda environment):
#conda install -n jupyter -c conda-forge r-nloptr r-lme4
library(data.table)
library(dplyr)
library(purrr)
library(lme4)
library(lmerTest)
library(broom)
library(broom.mixed)

DATA_DIR <- "../gp2_tier2_eu_release12"
OUT_DIR  <- "./rs1450522_GP2"

MASTER <- file.path(
  DATA_DIR,
  "../gp2_tier2_eu_release12/clinical_data/master_key_release12_final_vwb.csv"
)

CLIN <- file.path(
  DATA_DIR,
  "../gp2_tier2_eu_release12/clinical_data/r12_extended_clinical_data.csv"
)

In [ ]:
# Read clinical data
master <- fread(MASTER)
clin   <- fread(CLIN)

dim(master)
dim(clin)

names(master)
names(clin)

In [ ]:
# Because GP2 field naming has evolved across releases, resolve the handful of master-key columns programmatically
pick_col <- function(df, candidates) {

  hit <- candidates[candidates %in% names(df)]

  # if (length(hit) == 0) {
  #   stop(
  #     "None of these columns were found: ",
  #     paste(candidates, collapse = ", ")
  #   )
  # }

  hit[1]
}

sex_col <- pick_col(
  master,
  c("biological_sex_for_qc", "sex")
)

age_col <- pick_col(
  master,
  c("age_at_sample_collection", "age")
)

aao_col <- pick_col(
  master,
  c("age_of_onset")
)

pheno_col <- pick_col(
  master,
  c("GP2_PHENO")
)

study_type_col <- pick_col(
  master,
  c("study_type")
)

head(pick_col)
head(sex_col)
head(age_col)
head(aao_col)
head(pheno_col)
head(study_type_col)

### Then, genotype data

In [ ]:
# This code could be repurposed later in case of analyzing multiple ancestries instead of EUR only
geno_files <- list.files(
  file.path(OUT_DIR),
  pattern = "rs1450522.*_G\\.raw$",
  full.names = TRUE
)

geno <- map_dfr(geno_files, function(f) {

  x <- fread(f)

  fixed_cols <- c(
    "FID", "IID", "PAT",
    "MAT", "SEX", "PHENOTYPE"
  )

  dosage_col <- setdiff(names(x), fixed_cols)

  if (length(dosage_col) != 1) {
    stop("Unexpected dosage columns in ", f)
  }

  ancestry <- sub(
    "_rs1450522_G\\.raw$",
    "",
    basename(f)
  )

  tibble(
    GP2ID = as.character(x$IID),
    G_dosage = as.numeric(x[[dosage_col]]),
    ancestry = ancestry
  )
})

table(geno$G_dosage, useNA = "ifany")
table(geno$ancestry)

In [ ]:
# Inspect Hardy-Weinberg-ish counts and frequency
geno %>%
  group_by(ancestry) %>%
  summarise(
    n = n(),
    n_AA = sum(G_dosage == 0, na.rm = TRUE),
    n_AG = sum(G_dosage == 1, na.rm = TRUE),
    n_GG = sum(G_dosage == 2, na.rm = TRUE),
    G_AF = sum(G_dosage, na.rm = TRUE) /
      (2 * sum(!is.na(G_dosage)))
  )

## 3. Load principal components
Even if we're analyzing only EUR individuals, we still want the genetic PCs to control for within-European population structure.

In [ ]:
PC_FILE <- "../gp2_tier2_eu_release12/raw_genotypes/EUR/EUR_release12_vwb.eigenvec"

pcs <- fread(PC_FILE)

names(pcs)
head(pcs)

In [ ]:
pcs <- pcs %>%
    transmute(
        GP2ID = as.character(IID),
        PC1 = as.numeric(PC1),
        PC2 = as.numeric(PC2),
        PC3 = as.numeric(PC3),
        PC4 = as.numeric(PC4),
        PC5 = as.numeric(PC5),
        PC6 = as.numeric(PC6),
        PC7 = as.numeric(PC7),
        PC8 = as.numeric(PC8),
        PC9 = as.numeric(PC9),
        PC10 = as.numeric(PC10)
    )

In [ ]:
# Check overlap with rs1450522 genotypes
nrow(geno)
nrow(pcs)

sum(geno$GP2ID %in% pcs$GP2ID)
mean(geno$GP2ID %in% pcs$GP2ID)

## 4. Define PD cohort

In [ ]:
table(master$GP2_PHENO, useNA = "ifany")

In [ ]:
master_pd <- master %>%
    filter(
        GP2_PHENO %in% c("PD", "Affected_PD")
    )

In [ ]:
# Because release 12 naming may differ slightly, first check
grep(
    "age|sex|onset|diagnos|study|pheno",
    names(master),
    value = TRUE,
    ignore.case = TRUE
)

In [ ]:
# For longitudinal analysis, make sure to exclude brain-bank participants
master_pd <- master %>%
    filter(
        GP2_PHENO %in% c("PD", "Affected_PD")
    ) %>%
    transmute(
        GP2ID = as.character(GP2ID),

        sex = factor(biological_sex_for_qc),

        age = as.numeric(age_at_sample_collection),

        age_of_onset = as.numeric(age_of_onset),

        GP2_PHENO = GP2_PHENO,

        study_type = study_type
    ) %>%
    filter(
        is.na(study_type) |
        study_type != "Brain Bank"
    ) %>%
    mutate(
        disease_duration = ifelse(
            !is.na(age) &
            !is.na(age_of_onset) &
            age >= age_of_onset,
            age - age_of_onset,
            NA_real_
        )
    )

head(master)

## 5. Combine clinical, genotype, master-key and PC data

In [ ]:
# Inspect clinical ID and visit columns
grep(
    "GP2|visit|study",
    names(clin),
    value = TRUE,
    ignore.case = TRUE
)

In [ ]:
dat <- clin %>%
    mutate(
        GP2ID = as.character(GP2ID),
        visit_month = as.numeric(visit_month),
        time_years = visit_month / 12
    ) %>%

    # Restricts us to PD
    inner_join(
        master_pd,
        by = "GP2ID"
    ) %>%

    # Restricts us to EUR participants for whom we extracted rs1450522
    inner_join(
        geno,
        by = "GP2ID"
    ) %>%

    # Add EUR genetic PCs
    left_join(
        pcs,
        by = "GP2ID"
    )

In [ ]:
# Sanity checks
n_distinct(dat$GP2ID)

table(
    dat$G_dosage,
    useNA = "ifany"
)

summary(dat$age)
summary(dat$disease_duration)

In [ ]:
# Check missingness in PCs
colSums(
    is.na(
        dat[, paste0("PC", 1:10)]
    )
)

In [ ]:
# Check study representation
table(
    dat$study,
    useNA = "ifany"
)

In [ ]:
# Check genotype distribution at the participant level
# Don't use all clinical rows for this because people with multiple visits would be counted multiple times
dat %>%
    distinct(GP2ID, G_dosage) %>%
    count(G_dosage)

In [ ]:
# Calculate G allele frequency
geno_pd <- dat %>%
    distinct(GP2ID, G_dosage)

sum(geno_pd$G_dosage) /
    (2 * nrow(geno_pd))

## 6. Determine usable clinical phenotypes

In [ ]:
outcomes <- c(
    "MDS-UPDRS I"   = "mds_updrs_part_i_summary_score",
    "MDS-UPDRS II"  = "mds_updrs_part_ii_summary_score",
    "MDS-UPDRS III" = "mds_updrs_part_iii_summary_score",
    "MDS-UPDRS IV"  = "mds_updrs_part_iv_summary_score",
    "Hoehn-Yahr"    = "hoehn_and_yahr_stage",
    "MoCA"          = "moca_total_score",
    "MMSE"          = "mmse_total_score",
    "RBD"           = "rbd_summary_score",
    "ESS"           = "ess_total_score",
    "GDS15"         = "gds15_total_score"
)

In [ ]:
# See which outcomes actually exist in release 12
data.frame(
    phenotype = names(outcomes),
    variable = unname(outcomes),
    present = unname(outcomes) %in% names(dat)
)

In [ ]:
# Keep only available outcomes
outcomes <- outcomes[
    unname(outcomes) %in% names(dat)
]

In [ ]:
# Calculate coverage
# Before running associations, make sure sample sizes are decent
coverage <- map_dfr(
    names(outcomes),
    function(label) {

        v <- outcomes[[label]]

        d <- dat %>%
            filter(!is.na(.data[[v]]))

        baseline <- d %>%
            filter(visit_month == 0) %>%
            distinct(GP2ID, .keep_all = TRUE)

        longitudinal <- d %>%
            distinct(GP2ID, visit_month) %>%
            count(GP2ID) %>%
            filter(n >= 2)

        tibble(
            phenotype = label,
            variable = v,

            baseline_N =
                n_distinct(baseline$GP2ID),

            baseline_AA =
                sum(baseline$G_dosage == 0),

            baseline_AG =
                sum(baseline$G_dosage == 1),

            baseline_GG =
                sum(baseline$G_dosage == 2),

            longitudinal_N =
                nrow(longitudinal)
        )
    }
)

coverage

## 7. Baseline association analysis (additive model)
Model:
Y=β0​+β1​G+β2​Age+β3​Sex+β4​DiseaseDuration+∑PCk​+Study+ϵ

In [ ]:
# Use first 10 PCs
pc_cols <- paste0("PC", 1:10)

pc_cols <- pc_cols[
    pc_cols %in% names(dat)
]

In [ ]:
# Defining function
run_baseline <- function(outcome) {

    d <- dat %>%
        filter(
            visit_month == 0,
            !is.na(.data[[outcome]]),
            !is.na(G_dosage)
        ) %>%
        distinct(
            GP2ID,
            .keep_all = TRUE
        )

    covars <- c(
        "G_dosage",
        "age",
        "sex",
        "disease_duration",
        pc_cols,
        "factor(study)"
    )

    form <- reformulate(
        covars,
        response = outcome
    )

    fit <- lm(
        form,
        data = d
    )

    broom::tidy(
        fit,
        conf.int = TRUE
    ) %>%
        filter(term == "G_dosage") %>%
        mutate(
            outcome = outcome,
            N = nobs(fit)
        )
}

In [ ]:
# Run all outcomes
baseline_results <- map_dfr(
    unname(outcomes),
    run_baseline
)

In [ ]:
# FDR correction, and have a look at the results
baseline_results <- baseline_results %>%
    mutate(
        FDR = p.adjust(
            p.value,
            method = "BH"
        )
    ) %>%
    arrange(FDR)

baseline_results

For MDS-UPDRS:

    estimate > 0

means each additional G allele is associated with a higher/worse score.

For MoCA/MMSE:

    estimate < 0

would correspond to worse cognitive performance per G allele.

## 8. Longitudinal association
Question: Does rs1450522 affect the rate of progression?

Model:
Yij​=β0​+β1​G+β2​Time+β3​(G×Time)+Covariates+ui​+ϵij​

In [ ]:
# Defining function
run_longitudinal <- function(outcome) {

    d <- dat %>%
        filter(
            !is.na(.data[[outcome]]),
            !is.na(G_dosage),
            !is.na(visit_month),
            visit_month >= 0
        ) %>%
        group_by(GP2ID) %>%
        filter(
            n_distinct(visit_month) >= 2
        ) %>%
        ungroup()

    # Not enough participants for a meaningful model
    if (n_distinct(d$GP2ID) < 20) {
        message(
            outcome,
            ": skipped — fewer than 20 longitudinal participants"
        )
        return(NULL)
    }

    # ------------------------------------
    # Standardize continuous covariates
    # ------------------------------------

    d <- d %>%
        mutate(
            age_z = as.numeric(scale(age)),
            disease_duration_z =
                as.numeric(scale(disease_duration))
        )

    # Scale PCs as well
    for (pc in pc_cols) {

        new_name <- paste0(pc, "_z")

        d[[new_name]] <- as.numeric(
            scale(d[[pc]])
        )
    }

    pc_z_cols <- paste0(
        pc_cols,
        "_z"
    )

    # ------------------------------------
    # Build covariates dynamically
    # ------------------------------------

    covars <- c(
        "age_z",
        "disease_duration_z",
        pc_z_cols
    )

    # Only include sex if >=2 levels
    if (
        n_distinct(
            d$sex[!is.na(d$sex)]
        ) >= 2
    ) {
        covars <- c(
            covars,
            "sex"
        )
    }

    # Only include study if >=2 levels
    if (
        n_distinct(
            d$study[!is.na(d$study)]
        ) >= 2
    ) {
        covars <- c(
            covars,
            "factor(study)"
        )
    }

    # Remove covariates that became entirely NA
    covars <- covars[
        sapply(
            covars,
            function(v) {

                if (grepl("^factor\\(", v)) {
                    return(TRUE)
                }

                any(
                    !is.na(d[[v]])
                )
            }
        )
    ]

    fixed_part <- paste(
        covars,
        collapse = " + "
    )

    # ------------------------------------
    # Random slope model
    # ------------------------------------

    f <- as.formula(
        paste0(
            outcome,
            " ~ G_dosage * time_years + ",
            fixed_part,
            " + (1 + time_years | GP2ID)"
        )
    )

    fit <- lmer(
        f,
        data = d,
        REML = FALSE
    )

    # ------------------------------------
    # Fall back if singular
    # ------------------------------------

    if (isSingular(fit, tol = 1e-4)) {

        message(
            outcome,
            ": singular random-slope model; using random intercept"
        )

        f <- as.formula(
            paste0(
                outcome,
                " ~ G_dosage * time_years + ",
                fixed_part,
                " + (1 | GP2ID)"
            )
        )

        fit <- lmer(
            f,
            data = d,
            REML = FALSE
        )
    }

    # ------------------------------------
    # Extract SNP × time effect
    # ------------------------------------

    result <- broom.mixed::tidy(
        fit,
        effects = "fixed",
        conf.int = TRUE
    ) %>%
        filter(
            term == "G_dosage:time_years"
        ) %>%
        mutate(
            outcome = outcome,
            observations = nobs(fit),
            participants =
                n_distinct(d$GP2ID),
            studies =
                n_distinct(
                    d$study,
                    na.rm = TRUE
                )
        )

    return(result)
}

In [ ]:
# Running analysis
longitudinal_results <- map_dfr(
    unname(outcomes),
    run_longitudinal
)

longitudinal_results <- longitudinal_results %>%
    mutate(
        FDR = p.adjust(
            p.value,
            method = "BH"
        )
    ) %>%
    arrange(FDR)

longitudinal_results

# outcome <- outcomes[[9]]

# d_test <- dat %>%
#     filter(
#         !is.na(.data[[outcome]]),
#         !is.na(G_dosage),
#         !is.na(visit_month),
#         visit_month >= 0
#     ) %>%
#     group_by(GP2ID) %>%
#     filter(
#         n_distinct(visit_month) >= 2
#     ) %>%
#     ungroup()

# table(d_test$sex, useNA = "ifany")
# table(d_test$study, useNA = "ifany")

# n_distinct(d_test$sex, na.rm = TRUE)
# n_distinct(d_test$study, na.rm = TRUE)

For MDS-UPDRS:

    G_dosage:time_years > 0

means each G allele is associated with a faster annual increase in disease severity.

For MoCA:

    G_dosage:time_years < 0

means each G allele is associated with a faster annual cognitive decline.

## 9. Handle MDS-UPDRS III medication state

In [ ]:
table(
    dat$upd23b_clinical_state_on_medication,
    useNA = "ifany"
)

In [ ]:
# All observations
dat %>%
    filter(
        !is.na(
            mds_updrs_part_iii_summary_score
        )
    ) %>%
    count(
        upd23b_clinical_state_on_medication
    )

In [ ]:
# All subjects
dat %>%
    filter(
        !is.na(
            mds_updrs_part_iii_summary_score
        )
    ) %>%
    distinct(
        GP2ID,
        upd23b_clinical_state_on_medication
    ) %>%
    count(
        upd23b_clinical_state_on_medication
    )

In [ ]:
# Since there are more than 400 "OFF" subjects, we can define an OFF subset and make it the primary analysis
dat_m3_off <- dat %>%
    filter(
        upd23b_clinical_state_on_medication == "OFF"
    )

In [ ]:
run_baseline <- function(outcome, off_only = FALSE) {

    d <- dat %>%
        filter(
            visit_month == 0,
            !is.na(.data[[outcome]]),
            !is.na(G_dosage)
        )

    # For MDS-UPDRS III, optionally restrict to OFF-state assessments
    if (off_only) {

        d <- d %>%
            filter(
                upd23b_clinical_state_on_medication == "OFF"
            )
    }

    d <- d %>%
        distinct(
            GP2ID,
            .keep_all = TRUE
        )

    covars <- c(
        "G_dosage",
        "age",
        "sex",
        "disease_duration",
        pc_cols
    )

    # Only include study if >1 study remains after filtering
    if (
        n_distinct(
            d$study[!is.na(d$study)]
        ) >= 2
    ) {
        covars <- c(
            covars,
            "factor(study)"
        )
    }

    form <- reformulate(
        covars,
        response = outcome
    )

    fit <- lm(
        form,
        data = d
    )

    broom::tidy(
        fit,
        conf.int = TRUE
    ) %>%
        filter(
            term == "G_dosage"
        ) %>%
        mutate(
            outcome = outcome,
            N = nobs(fit),
            participants = n_distinct(d$GP2ID),
            off_only = off_only
        )
}

baseline_mds3_off <- run_baseline(
    outcome = "mds_updrs_part_iii_summary_score",
    off_only = TRUE
)

baseline_mds3_off

In [ ]:
run_longitudinal <- function(outcome) {

    d <- dat_m3_off %>%
        filter(
            !is.na(.data[[outcome]]),
            !is.na(G_dosage),
            !is.na(visit_month),
            visit_month >= 0
        ) %>%
        group_by(GP2ID) %>%
        filter(
            n_distinct(visit_month) >= 2
        ) %>%
        ungroup()

    # Not enough participants for a meaningful model
    if (n_distinct(d$GP2ID) < 20) {
        message(
            outcome,
            ": skipped — fewer than 20 longitudinal participants"
        )
        return(NULL)
    }

    # ------------------------------------
    # Standardize continuous covariates
    # ------------------------------------

    d <- d %>%
        mutate(
            age_z = as.numeric(scale(age)),
            disease_duration_z =
                as.numeric(scale(disease_duration))
        )

    # Scale PCs as well
    for (pc in pc_cols) {

        new_name <- paste0(pc, "_z")

        d[[new_name]] <- as.numeric(
            scale(d[[pc]])
        )
    }

    pc_z_cols <- paste0(
        pc_cols,
        "_z"
    )

    # ------------------------------------
    # Build covariates dynamically
    # ------------------------------------

    covars <- c(
        "age_z",
        "disease_duration_z",
        pc_z_cols
    )

    # Only include sex if >=2 levels
    if (
        n_distinct(
            d$sex[!is.na(d$sex)]
        ) >= 2
    ) {
        covars <- c(
            covars,
            "sex"
        )
    }

    # Only include study if >=2 levels
    if (
        n_distinct(
            d$study[!is.na(d$study)]
        ) >= 2
    ) {
        covars <- c(
            covars,
            "factor(study)"
        )
    }

    # Remove covariates that became entirely NA
    covars <- covars[
        sapply(
            covars,
            function(v) {

                if (grepl("^factor\\(", v)) {
                    return(TRUE)
                }

                any(
                    !is.na(d[[v]])
                )
            }
        )
    ]

    fixed_part <- paste(
        covars,
        collapse = " + "
    )

    # ------------------------------------
    # Random slope model
    # ------------------------------------

    f <- as.formula(
        paste0(
            outcome,
            " ~ G_dosage * time_years + ",
            fixed_part,
            " + (1 + time_years | GP2ID)"
        )
    )

    fit <- lmer(
        f,
        data = d,
        REML = FALSE
    )

    # ------------------------------------
    # Fall back if singular
    # ------------------------------------

    if (isSingular(fit, tol = 1e-4)) {

        message(
            outcome,
            ": singular random-slope model; using random intercept"
        )

        f <- as.formula(
            paste0(
                outcome,
                " ~ G_dosage * time_years + ",
                fixed_part,
                " + (1 | GP2ID)"
            )
        )

        fit <- lmer(
            f,
            data = d,
            REML = FALSE
        )
    }

    # ------------------------------------
    # Extract SNP × time effect
    # ------------------------------------

    result <- broom.mixed::tidy(
        fit,
        effects = "fixed",
        conf.int = TRUE
    ) %>%
        filter(
            term == "G_dosage:time_years"
        ) %>%
        mutate(
            outcome = outcome,
            observations = nobs(fit),
            participants =
                n_distinct(d$GP2ID),
            studies =
                n_distinct(
                    d$study,
                    na.rm = TRUE
                )
        )

    return(result)
}

longitudinal_results <- map_dfr(
    unname(outcomes),
    run_longitudinal
)

longitudinal_results <- longitudinal_results %>%
    mutate(
        FDR = p.adjust(
            p.value,
            method = "BH"
        )
    ) %>%
    arrange(FDR)

longitudinal_results

## 10. Plot graphs of suggestive associations

In [ ]:
# First, set the common genotype labels
library(dplyr)
library(ggplot2)

dat <- dat %>%
    mutate(
        genotype = factor(
            G_dosage,
            levels = c(0, 1, 2),
            labels = c("AA", "AG", "GG")
        )
    )

### Baseline MDS-UPDRS Part II — ON + OFF

In [ ]:
# Because the same participant can potentially appear more than once at a visit because of ON/OFF-related rows, 
# keep only one baseline observation per participant
plot_mds2_all <- dat %>%
    filter(
        visit_month == 0,
        !is.na(mds_updrs_part_ii_summary_score),
        !is.na(genotype)
    ) %>%
    distinct(
        GP2ID,
        .keep_all = TRUE
    )

p_mds2_all <- ggplot(
    plot_mds2_all,
    aes(
        x = genotype,
        y = mds_updrs_part_ii_summary_score
    )
) +
    geom_boxplot(
        width = 0.6,
        outlier.shape = NA
    ) +
    geom_jitter(
        width = 0.12,
        alpha = 0.35,
        size = 1.5
    ) +
    labs(
        x = "rs1450522 genotype",
        y = "MDS-UPDRS Part II score",
        title = "Baseline MDS-UPDRS Part II"
    ) +
    theme_classic(base_size = 14)

p_mds2_all

### Baseline MDS-UPDRS Part II - OFF only

In [ ]:
plot_mds2_off <- dat %>%
    filter(
        visit_month == 0,
        upd23b_clinical_state_on_medication == "OFF",
        !is.na(mds_updrs_part_ii_summary_score),
        !is.na(genotype)
    ) %>%
    distinct(
        GP2ID,
        .keep_all = TRUE
    )

p_mds2_off <- ggplot(
    plot_mds2_off,
    aes(
        x = genotype,
        y = mds_updrs_part_ii_summary_score
    )
) +
    geom_boxplot(
        width = 0.6,
        outlier.shape = NA
    ) +
    geom_jitter(
        width = 0.12,
        alpha = 0.35,
        size = 1.5
    ) +
    labs(
        x = "rs1450522 genotype",
        y = "MDS-UPDRS Part II score",
        title = "Baseline MDS-UPDRS Part II — OFF state"
    ) +
    theme_classic(base_size = 14)

p_mds2_off

### Longitudinal RBD — ON + OFF

In [ ]:
# Collapse repeated visits into one mean RBD score per participant
plot_rbd_long <- dat %>%
    filter(
        !is.na(rbd_summary_score),
        !is.na(genotype),
        !is.na(visit_month),
        visit_month >= 0
    ) %>%

    # Remove accidental duplicated participant-visit rows
    distinct(
        GP2ID,
        visit_month,
        .keep_all = TRUE
    ) %>%

    group_by(
        GP2ID,
        genotype
    ) %>%

    summarise(
        rbd_mean = mean(
            rbd_summary_score,
            na.rm = TRUE
        ),
        n_visits = n(),
        .groups = "drop"
    )

In [ ]:
# Optional - require at least 2 visits
plot_rbd_long <- plot_rbd_long %>%
    filter(n_visits >= 2)

In [ ]:
p_rbd_long <- ggplot(
    plot_rbd_long,
    aes(
        x = genotype,
        y = rbd_mean
    )
) +
    geom_boxplot(
        width = 0.6,
        outlier.shape = NA
    ) +
    geom_jitter(
        width = 0.12,
        alpha = 0.35,
        size = 1.5
    ) +
    labs(
        x = "rs1450522 genotype",
        y = "Mean longitudinal RBD score",
        title = "Longitudinal RBD"
    ) +
    theme_classic(base_size = 14)

p_rbd_long

### Longitudinal Hoehn & Yahr — OFF only

In [ ]:
plot_hy_off <- dat %>%
    filter(
        upd23b_clinical_state_on_medication == "OFF",
        !is.na(hoehn_and_yahr_stage),
        !is.na(genotype),
        !is.na(visit_month),
        visit_month >= 0
    ) %>%

    distinct(
        GP2ID,
        visit_month,
        .keep_all = TRUE
    ) %>%

    group_by(
        GP2ID,
        genotype
    ) %>%

    summarise(
        hy_mean = mean(
            hoehn_and_yahr_stage,
            na.rm = TRUE
        ),
        n_visits = n(),
        .groups = "drop"
    ) %>%

    filter(
        n_visits >= 2
    )

In [ ]:
p_hy_off <- ggplot(
    plot_hy_off,
    aes(
        x = genotype,
        y = hy_mean
    )
) +
    geom_boxplot(
        width = 0.6,
        outlier.shape = NA
    ) +
    geom_jitter(
        width = 0.12,
        alpha = 0.35,
        size = 1.5
    ) +
    labs(
        x = "rs1450522 genotype",
        y = "Mean longitudinal Hoehn & Yahr stage",
        title = "Longitudinal Hoehn & Yahr — OFF state"
    ) +
    theme_classic(base_size = 14)

p_hy_off

In [ ]:
# Sample sizes used
plot_mds2_all %>% count(genotype)
plot_mds2_off %>% count(genotype)
plot_rbd_long %>% count(genotype)
plot_hy_off %>% count(genotype)

### Trajectory plots for longitudinal analyses

In [ ]:
dat <- dat %>%
    mutate(
        genotype = factor(
            G_dosage,
            levels = c(0, 1, 2),
            labels = c("AA", "AG", "GG")
        )
    )

In [ ]:
# RBD
plot_rbd_traj <- dat %>%
    filter(
        !is.na(rbd_summary_score),
        !is.na(genotype),
        !is.na(time_years),
        visit_month >= 0
    ) %>%
    distinct(
        GP2ID,
        visit_month,
        .keep_all = TRUE
    ) %>%
    group_by(GP2ID) %>%
    filter(
        n_distinct(visit_month) >= 2
    ) %>%
    ungroup()

p_rbd_traj <- ggplot(
    plot_rbd_traj,
    aes(
        x = time_years,
        y = rbd_summary_score,
        group = GP2ID
    )
) +
    geom_line(
        alpha = 0.12,
        linewidth = 0.4
    ) +
    geom_point(
        alpha = 0.15,
        size = 1
    ) +
    geom_smooth(
        aes(
            group = genotype,
            linetype = genotype
        ),
        method = "lm",
        se = TRUE,
        linewidth = 1
    ) +
    labs(
        x = "Time since baseline (years)",
        y = "RBD summary score",
        linetype = "rs1450522 genotype",
        title = "Longitudinal RBD trajectories"
    ) +
    theme_classic(base_size = 14)

p_rbd_traj

# OPTIONAL - Linetypes + grayscale
# p_rbd_traj +
#     scale_linetype_manual(
#         values = c(
#             "AA" = "solid",
#             "AG" = "dashed",
#             "GG" = "dotted"
#         )
#     )

In [ ]:
# Hoehn & Yahr - Restricted to OFF state
plot_hy_traj <- dat %>%
    filter(
        upd23b_clinical_state_on_medication == "OFF",
        !is.na(hoehn_and_yahr_stage),
        !is.na(genotype),
        !is.na(time_years),
        visit_month >= 0
    ) %>%
    distinct(
        GP2ID,
        visit_month,
        .keep_all = TRUE
    ) %>%
    group_by(GP2ID) %>%
    filter(
        n_distinct(visit_month) >= 2
    ) %>%
    ungroup()

p_hy_traj <- ggplot(
    plot_hy_traj,
    aes(
        x = time_years,
        y = hoehn_and_yahr_stage,
        group = GP2ID
    )
) +
    geom_line(
        alpha = 0.12,
        linewidth = 0.4
    ) +
    geom_point(
        alpha = 0.15,
        size = 1
    ) +
    geom_smooth(
        aes(
            group = genotype,
            linetype = genotype
        ),
        method = "lm",
        se = TRUE,
        linewidth = 1
    ) +
    labs(
        x = "Time since baseline (years)",
        y = "Hoehn & Yahr stage",
        linetype = "rs1450522 genotype",
        title = "Longitudinal Hoehn & Yahr trajectories — OFF state"
    ) +
    theme_classic(base_size = 14)

p_hy_traj

## 11. Refining analysis for RBD
Most convincing association pattern so far was for rs1450522 status vs. RBD scores.
Because of that, I decided to refine the analysis.

In [ ]:
# ============================================================
# rs1450522 x longitudinal RBD analysis
# EUR ancestry only
# Outcome: RBD summary score (0-13)
# Primary model: beta-binomial mixed model
# ============================================================

library(dplyr)
library(ggplot2)
library(glmmTMB)
library(ggeffects)
library(broom.mixed)

# ------------------------------------------------------------
# 1. Prepare longitudinal RBD dataset
# ------------------------------------------------------------

rbd <- dat %>%
    filter(
        !is.na(rbd_summary_score),
        !is.na(G_dosage),
        !is.na(time_years),
        !is.na(GP2ID),
        visit_month >= 0
    ) %>%

    # One observation per participant per visit
    distinct(
        GP2ID,
        visit_month,
        .keep_all = TRUE
    ) %>%

    # Only participants with >=2 distinct visits
    group_by(GP2ID) %>%
    filter(
        n_distinct(visit_month) >= 2
    ) %>%
    ungroup() %>%

    mutate(
        genotype = factor(
            G_dosage,
            levels = c(0, 1, 2),
            labels = c("AA", "AG", "GG")
        )
    )


# ------------------------------------------------------------
# 2. Basic score sanity checks
# ------------------------------------------------------------

cat("\n========================================\n")
cat("RBD SCORE SANITY CHECKS\n")
cat("========================================\n\n")

cat("Observed score range:\n")
print(range(rbd$rbd_summary_score, na.rm = TRUE))

cat("\nExpected range: 0-13\n")

if (
    any(
        rbd$rbd_summary_score < 0 |
        rbd$rbd_summary_score > 13,
        na.rm = TRUE
    )
) {
    warning("RBD scores outside the expected 0-13 range were detected.")
}

cat("\nScore distribution:\n")
print(
    table(
        rbd$rbd_summary_score,
        useNA = "ifany"
    )
)


# ------------------------------------------------------------
# 3. Participant counts by genotype
# ------------------------------------------------------------

genotype_counts <- rbd %>%
    distinct(
        GP2ID,
        genotype
    ) %>%
    count(
        genotype,
        name = "participants"
    )

cat("\nParticipants by genotype:\n")
print(genotype_counts)


# ------------------------------------------------------------
# 4. Follow-up characteristics by genotype
# ------------------------------------------------------------

followup_summary <- rbd %>%
    group_by(
        GP2ID,
        genotype
    ) %>%
    summarise(
        n_visits = n_distinct(visit_month),
        max_followup = max(time_years),
        .groups = "drop"
    ) %>%
    group_by(genotype) %>%
    summarise(
        participants = n(),
        median_visits = median(n_visits),
        IQR_visits = IQR(n_visits),
        median_followup = median(max_followup),
        IQR_followup = IQR(max_followup),
        max_followup = max(max_followup),
        .groups = "drop"
    )

cat("\nFollow-up by genotype:\n")
print(followup_summary)


# ------------------------------------------------------------
# 5. Number of participants contributing over time
# ------------------------------------------------------------

rbd_time_counts <- rbd %>%
    mutate(
        year_bin = floor(time_years)
    ) %>%
    group_by(
        genotype,
        year_bin
    ) %>%
    summarise(
        N = n_distinct(GP2ID),
        .groups = "drop"
    )

cat("\nParticipants contributing by year:\n")
print(rbd_time_counts)


# ------------------------------------------------------------
# 6. Raw RBD summary by genotype
# ------------------------------------------------------------

raw_summary <- rbd %>%
    group_by(genotype) %>%
    summarise(
        observations = n(),
        participants = n_distinct(GP2ID),
        mean_score = mean(rbd_summary_score),
        sd_score = sd(rbd_summary_score),
        median_score = median(rbd_summary_score),
        IQR_score = IQR(rbd_summary_score),
        .groups = "drop"
    )

cat("\nRaw RBD scores by genotype:\n")
print(raw_summary)


# ------------------------------------------------------------
# 7. Standardize continuous covariates
#
# Do NOT scale genotype or time:
# genotype remains 0/1/2 copies of G
# time remains years
# ------------------------------------------------------------

rbd <- rbd %>%
    mutate(
        age_z = as.numeric(scale(age)),
        disease_duration_z =
            as.numeric(scale(disease_duration))
    )

# PCs that actually exist
pc_cols <- paste0("PC", 1:10)
pc_cols <- pc_cols[
    pc_cols %in% names(rbd)
]

for (pc in pc_cols) {

    rbd[[paste0(pc, "_z")]] <-
        as.numeric(
            scale(rbd[[pc]])
        )
}

pc_z_cols <- paste0(
    pc_cols,
    "_z"
)


# ------------------------------------------------------------
# 8. Construct covariate list
# ------------------------------------------------------------

covars <- c(
    "age_z",
    "disease_duration_z",
    pc_z_cols
)

# Include sex only if >=2 levels are represented
if (
    "sex" %in% names(rbd) &&
    n_distinct(
        rbd$sex[!is.na(rbd$sex)]
    ) >= 2
) {
    covars <- c(
        covars,
        "sex"
    )
}

# Include study only if multiple studies contribute
if (
    "study" %in% names(rbd) &&
    n_distinct(
        rbd$study[!is.na(rbd$study)]
    ) >= 2
) {
    covars <- c(
        covars,
        "factor(study)"
    )
}

cat("\nCovariates used:\n")
print(covars)


# ------------------------------------------------------------
# 9. Primary beta-binomial mixed model
#
# RBD score is treated as:
#
#   positive points = score
#   remaining points = 13 - score
#
# Beta-binomial allows extra dispersion relative to binomial.
#
# Main hypothesis:
#
#   G_dosage : time_years
#
# ------------------------------------------------------------

fixed_part <- paste(
    covars,
    collapse = " + "
)

formula_rbd <- as.formula(
    paste0(
        "cbind(rbd_summary_score, ",
        "13 - rbd_summary_score) ~ ",
        "G_dosage * time_years + ",
        fixed_part,
        " + (1 | GP2ID)"
    )
)

model_rbd <- glmmTMB(
    formula_rbd,
    data = rbd,
    family = betabinomial(
        link = "logit"
    )
)

cat("\n========================================\n")
cat("PRIMARY MODEL\n")
cat("========================================\n\n")

print(summary(model_rbd))


# ------------------------------------------------------------
# 10. Check convergence
# ------------------------------------------------------------

cat("\nModel convergence:\n")

convergence_ok <-
    model_rbd$fit$convergence == 0

cat(
    "Optimizer convergence code:",
    model_rbd$fit$convergence,
    "\n"
)

if (
    !is.null(
        model_rbd$sdr$pdHess
    )
) {
    cat(
        "Positive-definite Hessian:",
        model_rbd$sdr$pdHess,
        "\n"
    )
}

if (!convergence_ok) {
    warning(
        "Model did not converge cleanly."
    )
}


# ------------------------------------------------------------
# 11. Extract genotype x time result
# ------------------------------------------------------------

model_results <- broom.mixed::tidy(
    model_rbd,
    effects = "fixed",
    conf.int = TRUE,
    exponentiate = FALSE
)

interaction_result <- model_results %>%
    filter(
        term == "G_dosage:time_years"
    )

cat("\nGenotype x time interaction:\n")
print(interaction_result)


# ------------------------------------------------------------
# 12. Convert interaction to odds ratio
#
# In this model, exp(beta) is the multiplicative change in
# odds of a higher RBD score trajectory per:
#
#     +1 G allele
#     +1 year
#
# ------------------------------------------------------------

interaction_result <- interaction_result %>%
    mutate(
        OR = exp(estimate),
        OR_low = exp(conf.low),
        OR_high = exp(conf.high)
    )

cat("\nInteraction as odds ratio:\n")
print(
    interaction_result %>%
        select(
            estimate,
            conf.low,
            conf.high,
            OR,
            OR_low,
            OR_high,
            p.value
        )
)


# ------------------------------------------------------------
# 13. Model-predicted trajectories
#
# Generate predicted scores for AA / AG / GG over follow-up.
# ------------------------------------------------------------

max_plot_time <- min(
    20,
    max(
        rbd$time_years,
        na.rm = TRUE
    )
)

prediction_grid <- ggpredict(
    model_rbd,
    terms = c(
        paste0(
            "time_years [0:",
            max_plot_time,
            " by=0.25]"
        ),
        "G_dosage [0,1,2]"
    ),
    type = "fixed"
)

pred <- as.data.frame(
    prediction_grid
)

pred <- pred %>%
    mutate(
        genotype = factor(
            group,
            levels = c("0", "1", "2"),
            labels = c("AA", "AG", "GG")
        ),

        # ggeffects gives predicted proportion for the
        # beta-binomial response; multiply by 13
        predicted_score =
            predicted * 13,

        conf.low.score =
            conf.low * 13,

        conf.high.score =
            conf.high * 13
    )


# ------------------------------------------------------------
# 14. Observed mean scores in 2-year bins
#
# These are shown only as descriptive data behind the model.
# ------------------------------------------------------------

observed_bins <- rbd %>%
    mutate(
        time_bin =
            floor(time_years / 2) * 2
    ) %>%

    group_by(
        genotype,
        time_bin
    ) %>%

    summarise(
        mean_score =
            mean(rbd_summary_score),

        N =
            n_distinct(GP2ID),

        .groups = "drop"
    )


# ------------------------------------------------------------
# 15. Prepare statistics for figure
# ------------------------------------------------------------

beta_int <-
    interaction_result$estimate

ci_low <-
    interaction_result$conf.low

ci_high <-
    interaction_result$conf.high

p_int <-
    interaction_result$p.value

OR_int <-
    interaction_result$OR

OR_low <-
    interaction_result$OR_low

OR_high <-
    interaction_result$OR_high


p_label <- ifelse(
    p_int < 0.001,
    "P < 0.001",
    paste0(
        "P = ",
        formatC(
            p_int,
            format = "f",
            digits = 3
        )
    )
)

stats_label <- paste0(
    "Genotype × time: ",
    "β = ",
    round(beta_int, 3),
    " (95% CI ",
    round(ci_low, 3),
    " to ",
    round(ci_high, 3),
    "); ",
    "OR = ",
    round(OR_int, 2),
    " (",
    round(OR_low, 2),
    "–",
    round(OR_high, 2),
    "); ",
    p_label
)


# ------------------------------------------------------------
# 16. Publication-style trajectory figure
# ------------------------------------------------------------

p_rbd_model <- ggplot() +

    # Observed binned means
    geom_point(
        data = observed_bins,
        aes(
            x = time_bin,
            y = mean_score,
            shape = genotype
        ),
        alpha = 0.45,
        size = 2
    ) +

    # 95% CI ribbons
    geom_ribbon(
        data = pred,
        aes(
            x = x,
            ymin = conf.low.score,
            ymax = conf.high.score,
            group = genotype
        ),
        alpha = 0.12
    ) +

    # Model-predicted trajectories
    geom_line(
        data = pred,
        aes(
            x = x,
            y = predicted_score,
            linetype = genotype,
            group = genotype
        ),
        linewidth = 1.1
    ) +

    scale_linetype_manual(
        values = c(
            "AA" = "solid",
            "AG" = "dashed",
            "GG" = "dotted"
        )
    ) +

    labs(
        x = "Time since baseline (years)",
        y = "Predicted RBD summary score",

        title =
            "rs1450522 genotype and longitudinal RBD severity",

        subtitle =
            stats_label,

        linetype =
            "rs1450522 genotype",

        shape =
            "Observed genotype",

        caption =
            paste0(
                "Lines and shaded areas show beta-binomial mixed-model ",
                "predictions and 95% CIs. Points show observed mean ",
                "RBD scores in 2-year bins."
            )
    ) +

    coord_cartesian(
        ylim = c(0, 13)
    ) +

    theme_classic(
        base_size = 13
    ) +

    theme(
        plot.title =
            element_text(
                face = "bold"
            ),

        plot.subtitle =
            element_text(
                size = 10
            ),

        legend.position =
            "right"
    )

p_rbd_model

In [ ]:
# ============================================================
# 17. Automated interpretation
# ============================================================

cat("\n\n")
cat("========================================\n")
cat("FINAL INTERPRETATION\n")
cat("========================================\n\n")

direction <- ifelse(
    beta_int > 0,
    "faster increase in RBD severity",
    "slower increase in RBD severity"
)

if (
    p_int < 0.05 &&
    ci_low > 0
) {

    conclusion <- paste0(
        "There is evidence that rs1450522 G dosage is associated ",
        "with longitudinal RBD progression in this EUR GP2 cohort. ",
        "Each additional G allele was associated with a ",
        direction,
        " over time (genotype × time beta = ",
        round(beta_int, 3),
        ", 95% CI ",
        round(ci_low, 3),
        " to ",
        round(ci_high, 3),
        "; OR = ",
        round(OR_int, 2),
        ", 95% CI ",
        round(OR_low, 2),
        " to ",
        round(OR_high, 2),
        "; ",
        p_label,
        ")."
    )

} else if (
    p_int < 0.05 &&
    ci_high < 0
) {

    conclusion <- paste0(
        "There is evidence that rs1450522 G dosage is associated ",
        "with longitudinal RBD progression in this EUR GP2 cohort. ",
        "Each additional G allele was associated with a ",
        direction,
        " over time (genotype × time beta = ",
        round(beta_int, 3),
        ", 95% CI ",
        round(ci_low, 3),
        " to ",
        round(ci_high, 3),
        "; OR = ",
        round(OR_int, 2),
        ", 95% CI ",
        round(OR_low, 2),
        " to ",
        round(OR_high, 2),
        "; ",
        p_label,
        ")."
    )

} else {

    conclusion <- paste0(
        "The analysis does not provide statistically clear evidence ",
        "that rs1450522 G dosage is associated with longitudinal ",
        "RBD progression in this EUR GP2 cohort. ",
        "The genotype × time estimate was ",
        round(beta_int, 3),
        " (95% CI ",
        round(ci_low, 3),
        " to ",
        round(ci_high, 3),
        "; OR = ",
        round(OR_int, 2),
        ", 95% CI ",
        round(OR_low, 2),
        " to ",
        round(OR_high, 2),
        "; ",
        p_label,
        ")."
    )
}

cat(conclusion)
cat("\n")

Note that the old longitudinal analysis showed a possible positive association of the G allele with RBD scores (if you have the PD risk allele, you have more severe sleep disorder).
But the new fine-tuned analysis showed the opposite pattern (if you have the PD risk allele, you have less severe sleep disorder).

Because of this discordance, we should be suspicious about the validity of this result...

## 12. Analyzing additional phenotypes

### Testing whether genotype-by-time effect differs by biological sex

In [ ]:
formula_rbd_sex <- as.formula(
    paste0(
        "cbind(rbd_summary_score, 13 - rbd_summary_score) ~ ",
        "G_dosage * time_years * sex + ",
        "age_z + disease_duration_z + ",
        paste(pc_z_cols, collapse = " + "),
        " + factor(study) + (1 | GP2ID)"
    )
)

model_rbd_sex <- glmmTMB(
    formula_rbd_sex,
    data = rbd,
    family = betabinomial(link = "logit")
)

summary(model_rbd_sex)

In [ ]:
# Small sex interaction effect. Check sex sample sizes.
rbd %>%
    distinct(GP2ID, sex, genotype) %>%
    count(sex, genotype)

In [ ]:
# Stratify by sex
rbd_female <- rbd %>% filter(sex == "Female")
rbd_male   <- rbd %>% filter(sex == "Male")

In [ ]:
# Sex-stratified analysis
run_rbd_by_sex <- function(sex_value) {

    d <- rbd %>%
        filter(
            sex == sex_value
        ) %>%
        droplevels()

    # Only include study if multiple studies remain
    covars <- c(
        "age_z",
        "disease_duration_z",
        pc_z_cols
    )

    if (
        n_distinct(
            d$study[!is.na(d$study)]
        ) >= 2
    ) {
        covars <- c(
            covars,
            "factor(study)"
        )
    }

    fixed_part <- paste(
        covars,
        collapse = " + "
    )

    f <- as.formula(
        paste0(
            "cbind(rbd_summary_score, 13 - rbd_summary_score) ~ ",
            "G_dosage * time_years + ",
            fixed_part,
            " + (1 | GP2ID)"
        )
    )

    fit <- glmmTMB(
        f,
        data = d,
        family = betabinomial(
            link = "logit"
        )
    )

    result <- broom.mixed::tidy(
        fit,
        effects = "fixed",
        conf.int = TRUE
    ) %>%
        filter(
            term == "G_dosage:time_years"
        ) %>%
        mutate(
            sex = sex_value,
            OR = exp(estimate),
            OR_low = exp(conf.low),
            OR_high = exp(conf.high),
            participants = n_distinct(d$GP2ID),
            observations = nrow(d)
        )

    list(
        model = fit,
        data = d,
        result = result
    )
}

rbd_female <- run_rbd_by_sex("Female")
rbd_male   <- run_rbd_by_sex("Male")

sex_results <- bind_rows(
    rbd_female$result,
    rbd_male$result
)

sex_results

In [ ]:
# Sex-stratified plotting
make_predictions <- function(x) {

    max_time <- min(
        15,
        max(
            x$data$time_years,
            na.rm = TRUE
        )
    )

    pred <- ggpredict(
        x$model,
        terms = c(
            paste0(
                "time_years [0:",
                max_time,
                " by=0.25]"
            ),
            "G_dosage [0,1,2]"
        ),
        type = "fixed",
        bias_correction = TRUE
    )

    pred <- as.data.frame(pred)

    pred %>%
        mutate(
            genotype = factor(
                group,
                levels = c("0", "1", "2"),
                labels = c("AA", "AG", "GG")
            ),

            predicted_score =
                predicted * 13,

            conf.low.score =
                conf.low * 13,

            conf.high.score =
                conf.high * 13
        )
}

pred_female <- make_predictions(
    rbd_female
)

pred_male <- make_predictions(
    rbd_male
)

make_observed_bins <- function(d) {

    d %>%
        mutate(
            time_bin =
                floor(time_years / 2) * 2
        ) %>%
        group_by(
            genotype,
            time_bin
        ) %>%
        summarise(
            mean_score =
                mean(
                    rbd_summary_score,
                    na.rm = TRUE
                ),

            N =
                n_distinct(GP2ID),

            .groups = "drop"
        )
}

obs_female <- make_observed_bins(
    rbd_female$data
)

obs_male <- make_observed_bins(
    rbd_male$data
)

plot_rbd_sex <- function(
    pred,
    obs,
    result,
    sex_label
) {

    beta <- result$estimate
    low  <- result$conf.low
    high <- result$conf.high
    pval <- result$p.value

    p_label <- ifelse(
        pval < 0.001,
        "P < 0.001",
        paste0(
            "P = ",
            formatC(
                pval,
                format = "f",
                digits = 3
            )
        )
    )

    stats_label <- paste0(
        "G dosage × time: β = ",
        round(beta, 3),
        " (95% CI ",
        round(low, 3),
        " to ",
        round(high, 3),
        "); ",
        p_label
    )

    ggplot() +

        # Observed 2-year-bin means
        geom_point(
            data = obs,
            aes(
                x = time_bin,
                y = mean_score,
                shape = genotype
            ),
            size = 2,
            alpha = 0.45
        ) +

        # Model confidence intervals
        geom_ribbon(
            data = pred,
            aes(
                x = x,
                ymin = conf.low.score,
                ymax = conf.high.score,
                group = genotype
            ),
            alpha = 0.12
        ) +

        # Model-predicted trajectories
        geom_line(
            data = pred,
            aes(
                x = x,
                y = predicted_score,
                linetype = genotype,
                group = genotype
            ),
            linewidth = 1.1
        ) +

        scale_linetype_manual(
            values = c(
                "AA" = "solid",
                "AG" = "dashed",
                "GG" = "dotted"
            )
        ) +

        coord_cartesian(
            ylim = c(0, 13)
        ) +

        labs(
            x = "Time since baseline (years)",
            y = "Predicted RBD summary score",

            title = paste0(
                "rs1450522 and longitudinal RBD — ",
                sex_label
            ),

            subtitle = stats_label,

            linetype =
                "rs1450522 genotype",

            shape =
                "Observed genotype"
        ) +

        theme_classic(
            base_size = 14
        ) +

        theme(
            plot.title =
                element_text(
                    face = "bold"
                )
        )
}

p_female <- plot_rbd_sex(
    pred_female,
    obs_female,
    rbd_female$result,
    "Female"
)

p_male <- plot_rbd_sex(
    pred_male,
    obs_male,
    rbd_male$result,
    "Male"
)

# p_female
# p_male
library(patchwork)

p_sex <- p_female + p_male +
    plot_layout(
        guides = "collect"
    ) &
    theme(
        legend.position = "bottom"
    )

p_sex

In [ ]:
# Note that one dot represents the mean RBD score of potentially dozens or hundreds of participants in that genotype/time bin, not one participant.
# Replotting with better scale:
options(
    repr.plot.width = 14,
    repr.plot.height = 8
)

p_sex

In [ ]:
ggsave(
    "rs1450522_RBD_by_sex.pdf",
    p_sex,
    width = 14,
    height = 8,
    dpi = 600
)

### Genotype vs. age of onset

In [ ]:
aao_dat <- master %>%
    filter(
        GP2_PHENO %in% c("PD", "Affected_PD"),
        !is.na(age_of_onset)
    ) %>%
    inner_join(geno, by = "GP2ID") %>%
    left_join(pcs, by = "GP2ID")

fit_aao <- lm(
    age_of_onset ~
        G_dosage +
        biological_sex_for_qc +
        PC1 + PC2 + PC3 + PC4 + PC5 +
        PC6 + PC7 + PC8 + PC9 + PC10 +
        factor(study),
    data = aao_dat
)

summary(fit_aao)
confint(fit_aao)["G_dosage", ]

In [ ]:
# Plotting
library(dplyr)
library(ggplot2)
library(broom)

aao_dat <- aao_dat %>%
    mutate(
        genotype = factor(
            G_dosage,
            levels = c(0, 1, 2),
            labels = c("AA", "AG", "GG")
        )
    )

# Fit adjusted model
fit_aao <- lm(
    age_of_onset ~
        G_dosage +
        biological_sex_for_qc +
        PC1 + PC2 + PC3 + PC4 + PC5 +
        PC6 + PC7 + PC8 + PC9 + PC10 +
        factor(study),
    data = aao_dat
)

aao_result <- broom::tidy(
    fit_aao,
    conf.int = TRUE
) %>%
    filter(term == "G_dosage")

aao_result

# Create statistics label
beta <- aao_result$estimate
ci_low <- aao_result$conf.low
ci_high <- aao_result$conf.high
pval <- aao_result$p.value

p_label <- ifelse(
    pval < 0.001,
    "P < 0.001",
    paste0("P = ", formatC(pval, format = "f", digits = 3))
)

stats_label <- paste0(
    "\u03b2 = ", round(beta, 2),
    " years/G allele (95% CI ",
    round(ci_low, 2), " to ", round(ci_high, 2),
    "); ", p_label
)

# Plot
p_aao <- ggplot(
    aao_dat,
    aes(
        x = genotype,
        y = age_of_onset
    )
) +
    geom_boxplot(
        width = 0.6,
        outlier.shape = NA
    ) +
    geom_jitter(
        width = 0.12,
        alpha = 0.25,
        size = 1.3
    ) +
    labs(
        x = "rs1450522 genotype",
        y = "Age at PD onset (years)",
        title = "rs1450522 genotype and Parkinson's disease age of onset",
        subtitle = stats_label
    ) +
    theme_classic(base_size = 14) +
    theme(
        plot.title = element_text(face = "bold")
    )

p_aao